# HT/NALM vs HT/HB — CD4 helper view

Both cell systems share the same **healthy T donor**; the **B target** differs:

* **HT/NALM** = `NALM-6 + healthy T`     — B-ALL cell-line target
* **HT/HB**   = `healthy B + healthy T`  — primary healthy B cells

Because the T compartment is held fixed, the analyses below ask:
**does the healthy CD4 helper compartment respond differently to the NALM-6
cell line vs primary healthy B cells?** Helper signalling (CD40L, ICOS, OX40)
is the focus, with CD4-relevant checkpoint biology (PD-1, CTLA-4, TIGIT,
TIM-3) tracked alongside. CD4 helper T cells engage targets through **MHC-II**
(HLA-DR/DP/DQ), unlike the MHC-I view relevant for CD8.

Sample availability:

| Time × Condition | HT/NALM | HT/HB |
| --- | --- | --- |
| 6h Mock          | S005 | S001 |
| 6h Blinatumomab  | S006 | S002 |
| 48h Mock         | S007 | S003 |
| 48h Blinatumomab | S008 | S004 |

Cross-system comparisons are run at **6h** (consistent with the other paired-system notebooks).

In [ ]:
# [0 · Imports & configuration]
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import scanpy as sc
import scvi
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests

sc.set_figure_params(dpi=100, frameon=False)
plt.rcParams['figure.max_open_warning'] = 0

print(f'scvi-tools: {scvi.__version__}')

from nalm_utils import *

CACHE_DIR        = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data/cache')
ANNOTATED_CACHE  = CACHE_DIR / 'adata_cytovi_annotated_compat.h5ad'

SYS_HT_NALM = 'NALM-6 + healthy T'       # HT/NALM
SYS_HT_HB   = 'healthy B + healthy T'    # HT/HB

In [ ]:
# [1 · Data loading]
adata = sc.read_h5ad(ANNOTATED_CACHE)

mask_sys = adata.obs['cell_system'].isin([SYS_HT_NALM, SYS_HT_HB])
print(f'Total cells in HT/NALM + HT/HB: {mask_sys.sum()}')
pd.crosstab(
    index=[adata.obs[mask_sys]['cell_system'], adata.obs[mask_sys]['sample']],
    columns=[adata.obs[mask_sys]['time'], adata.obs[mask_sys]['condition']],
)

In [ ]:
# [1b · UMAP exploration]
mask_explore = (
    (adata.obs['cell_type_annot'].isin(['CD4', 'CD8', 'B'])) &
    (adata.obs['cell_system'].isin([SYS_HT_NALM, SYS_HT_HB]))
)
adata_sub = adata[mask_explore].copy()

print(f'Cells in HT/NALM + HT/HB (CD4/CD8/B): {adata_sub.n_obs}')
pd.crosstab(index=adata_sub.obs['sample'],
            columns=[adata_sub.obs['cell_system'], adata_sub.obs['cell_type_annot']])

sc.pl.umap(adata_sub,
  color=['CD3e', 'CD4', 'CD19', 'cell_system', 'cell_type_annot', 'condition'],
  layer='arcsinh', frameon=False)

## Cross-condition comparison — CD4 across time × condition

In [ ]:
# [2 · 4-way DA panel: CD4 — HT/NALM vs HT/HB across time × condition]
# B cell markers are dropped to keep the focus on T-cell biology.
def _load_b_panel():
    for name in ('b_cell_markers', 'or'):
        try:
            return load_marker_panel(name)
        except KeyError:
            continue
    raise KeyError('No B cell marker panel found in marker_panels.json')

B_CELL_MARKERS_SET = set(_load_b_panel())

mask_nm_cd4 = (
    (adata.obs['cell_system'] == SYS_HT_NALM) &
    (adata.obs['cell_type_annot'] == 'CD4')
)
adata_nm_cd4 = adata[mask_nm_cd4].copy()
adata_nm_cd4.obs['time_cond'] = (
    adata_nm_cd4.obs['time'].astype(str) + ' ' + adata_nm_cd4.obs['condition'].astype(str)
)
adata_nm_cd4 = adata_nm_cd4[:, ~adata_nm_cd4.var_names.isin(B_CELL_MARKERS_SET)].copy()

mask_hb_cd4 = (
    (adata.obs['cell_system'] == SYS_HT_HB) &
    (adata.obs['cell_type_annot'] == 'CD4')
)
adata_hb_cd4 = adata[mask_hb_cd4].copy()
adata_hb_cd4.obs['time_cond'] = (
    adata_hb_cd4.obs['time'].astype(str) + ' ' + adata_hb_cd4.obs['condition'].astype(str)
)
adata_hb_cd4 = adata_hb_cd4[:, ~adata_hb_cd4.var_names.isin(B_CELL_MARKERS_SET)].copy()

print(f'CD4 HT/NALM: {adata_nm_cd4.n_obs}')
print(adata_nm_cd4.obs['time_cond'].value_counts().to_string())
print(f'\nCD4 HT/HB: {adata_hb_cd4.n_obs}')
print(adata_hb_cd4.obs['time_cond'].value_counts().to_string())

plot_marker_panel_violins(
    adata_nm_cd4, 'cd4_t_cell_markers',
    group_key='time_cond',
    adata_compare=adata_hb_cd4,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

## LFC scatter — Blinatumomab vs Mock at 6h

In [ ]:
# [7 · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), CD4, 6h — top-K farthest from y=x]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD4',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd',  # higher LFC in HT/NALM
    color_below='#2ca02c',  # higher LFC in HT/HB
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# [7' · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), CD4, 6h — selected markers]

# Markers of interest (modify freely):
markers_of_interest = [
    'CD134',  # OX40 (TNFRSF4) — costimulatory; activated/memory CD4, Tfh, Treg
    'CD137',  # 4-1BB (TNFRSF9) — inducible costim; antigen-activated T (CD4/CD8), Treg
    'NKp80',  # KLRF1 — activating C-type lectin; NK / NKT (rare on CD4)
    'NKG2C',  # KLRC2 (CD159c) — activating NK receptor; CMV-adaptive NK, some CD8 (atypical on CD4)
    'NKp30',  # NCR3 (CD337) — natural cytotoxicity receptor; NK (atypical on CD4)
    'NKp46',  # NCR1 (CD335) — pan-NK marker; rare ILC1/T subsets
    'CD9',    # tetraspanin — activated/tissue-resident T, Treg-like
    'CD81',   # tetraspanin — TCR/CD19 partner; on T pairs with CD4 in synapse
    'VISTA',  # B7-H5 / PD-1H — inhibitory checkpoint; mainly myeloid, also on T
    'TIGIT',  # co-inhibitory checkpoint; activated CD4/CD8, Treg, Tfh
    'CD357',  # GITR (TNFRSF18) — costimulatory; high on Treg, induced on activated CD4
]

fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD4',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd',  # higher LFC in HT/NALM
    color_below='#2ca02c',  # higher LFC in HT/HB
    highlight_markers=markers_of_interest,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), CD4 spatial pairs, 6h

Companion to [7] applied to **spatial colocalization pairs** rather than
abundance. Mean difference (Blina − Mock) is used rather than log2 ratio
since coloc values can be negative (repulsion). Points above y=x: stronger
Blina-induced colocalization shift in HT/NALM (purple); below y=x: stronger
shift in HT/HB (green).

In [ ]:
# [7b · Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), CD4, 6h]
def plot_spatial_diff_scatter(
    adata_scope, time_val, cond_num, cond_den, sys_a, sys_b, cell_type,
    label_a, label_b, color_above, color_below,
    top_k=10, obsm_key='spatial_asinh5', ax=None,
):
    """Per-pair (cond_num − cond_den) spatial mean-diff scatter, sys_a (x) vs sys_b (y)."""
    def _diff(system_val):
        mask = (
            (adata_scope.obs['time'] == time_val) &
            (adata_scope.obs['cell_type_annot'] == cell_type) &
            (adata_scope.obs['cell_system'] == system_val)
        )
        sub = adata_scope[mask]
        sp = sub.obsm[obsm_key]
        if not isinstance(sp, pd.DataFrame):
            sp = pd.DataFrame(sp, index=sub.obs_names)
        cond = sub.obs['condition'].values
        num = sp.values[cond == cond_num].mean(axis=0)
        den = sp.values[cond == cond_den].mean(axis=0)
        return pd.Series(num - den, index=sp.columns)

    a = _diff(sys_a)
    b = _diff(sys_b)
    df = pd.DataFrame({'a': a, 'b': b}).replace([np.inf, -np.inf], np.nan).dropna()
    df['off_diag'] = (df['b'] - df['a']) / np.sqrt(2)

    top_idx = df['off_diag'].abs().nlargest(top_k).index
    colors = np.where(df.loc[top_idx, 'off_diag'] > 0, color_above, color_below)
    color_map = dict(zip(top_idx, colors))

    if ax is None:
        _, ax = plt.subplots(figsize=(8, 8))

    lim = max(df[['a', 'b']].abs().max().max() * 1.15, 0.1)
    ax.axhline(0, color='grey', lw=0.7, ls='--')
    ax.axvline(0, color='grey', lw=0.7, ls='--')
    ax.plot([-lim, lim], [-lim, lim], color='grey', lw=0.7, ls=':')

    other_idx = df.index.difference(top_idx)
    ax.scatter(df.loc[other_idx, 'a'], df.loc[other_idx, 'b'],
               s=30, alpha=0.55, color='#bbbbbb', edgecolor='white', linewidth=0.5)
    ax.scatter(df.loc[top_idx, 'a'], df.loc[top_idx, 'b'],
               s=70, alpha=0.9, c=[color_map[m] for m in top_idx],
               edgecolor='black', linewidth=0.6, zorder=3)

    def _pair_label(p):
        x, y = split_pair(p)
        return f'{display_name(x)} / {display_name(y)}'

    for m in top_idx:
        ax.annotate(_pair_label(m), (df.loc[m, 'a'], df.loc[m, 'b']),
                    fontsize=9, fontweight='bold', color=color_map[m],
                    xytext=(4, 4), textcoords='offset points')

    r = df['a'].corr(df['b'])
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    ax.set_xlabel(f'Mean diff {cond_num} − {cond_den}\n{label_a}')
    ax.set_ylabel(f'Mean diff {cond_num} − {cond_den}\n{label_b}')
    ax.set_title(f'{cell_type} spatial pairs — {time_val}   (n={len(df)}, Pearson r={r:.2f})')

    print(f'\n{"=" * 65}')
    print(f'  {cell_type} {time_val}  —  top {top_k} pairs farthest from y=x')
    print(f'  {color_above} = higher diff in {label_b} | {color_below} = higher diff in {label_a}')
    print(f'{"=" * 65}')
    top_tbl = df.loc[top_idx, ['a', 'b', 'off_diag']].copy()
    top_tbl.index = [_pair_label(m) for m in top_tbl.index]
    top_tbl.columns = [f'diff_{label_a}', f'diff_{label_b}', 'off_diag']
    top_tbl = top_tbl.reindex(top_tbl['off_diag'].abs().sort_values(ascending=False).index)
    print(top_tbl.round(3).to_string())
    return df


fig, ax = plt.subplots(figsize=(8, 8))
plot_spatial_diff_scatter(
    adata, time_val='6h',
    cond_num='Blinatumomab', cond_den='Mock',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD4',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd', color_below='#2ca02c',
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Mock vs Blinatumomab — abundance + spatial, per system

In [ ]:
# [8 · CD4 Blina vs Mock — abundance + spatial, per system, 6h]
for sys_label, sys_val in [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)]:
    plot_condition_comparison(
        adata,
        time_val='6h',
        cell_system=sys_val,
        cell_type='CD4',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

## MA plots — LFC vs mean abundance, CD4, 6h

Mean signal (x, averaged across both conditions) vs Blina − Mock LFC (y). Red
= FDR < 0.05 up in Blina, blue = FDR < 0.05 up in Mock, grey = n.s.;
top features by |LFC| among significant labelled.

In [ ]:
# [8b · MA plots — Blina vs Mock LFC vs mean abundance, per system, CD4, 6h]
ma_results_cd4 = plot_ma_blina_vs_mock(
    adata,
    systems=[('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)],
    time_val='6h',
    cell_type='CD4',
)

## ΔLFC vs mean abundance — HT/NALM vs HT/HB, CD4, 6h

*LFC of LFC.* ΔLFC = LFC(HT/NALM) − LFC(HT/HB) — how much *more* the CD4
helper response in the NALM-6 system exceeds the response with healthy
primary B targets. Same healthy T donor in both, so ΔLFC isolates the
contribution of the B target.

In [ ]:
# [8c · ΔLFC (LFC of LFC) vs mean signal — CD4, 6h, NALM vs HB]
delta_ab_cd4, delta_sp_cd4 = plot_delta_lfc_ma(
    ma_results_cd4,
    label_a='HT/NALM',
    label_b='HT/HB',
    cell_type='CD4',
    time_val='6h',
)

## Spatial subsets for selected-marker comparisons

In [ ]:
# [9 · Spatial subsets — CD4 cells, per condition / system]
def _sp_subset(adata_full, time_val, cond_val, system_val):
    mask = (
        (adata_full.obs['time'] == time_val) &
        (adata_full.obs['condition'] == cond_val) &
        (adata_full.obs['cell_type_annot'] == 'CD4') &
        (adata_full.obs['cell_system'] == system_val)
    )
    sub = adata_full[mask]
    sp = sub.obsm['spatial_asinh5']
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    print(f'  {time_val} {cond_val:15s} {system_val:25s} → {sp.shape[0]} cells')
    return sp

print('Building spatial subsets (CD4 only):')
sp_6h_mock_nm    = _sp_subset(adata, '6h',  'Mock',         SYS_HT_NALM)
sp_6h_mock_hb    = _sp_subset(adata, '6h',  'Mock',         SYS_HT_HB)
sp_6h_blina_nm   = _sp_subset(adata, '6h',  'Blinatumomab', SYS_HT_NALM)
sp_6h_blina_hb   = _sp_subset(adata, '6h',  'Blinatumomab', SYS_HT_HB)
sp_48h_mock_nm   = _sp_subset(adata, '48h', 'Mock',         SYS_HT_NALM)
sp_48h_mock_hb   = _sp_subset(adata, '48h', 'Mock',         SYS_HT_HB)
sp_48h_blina_nm  = _sp_subset(adata, '48h', 'Blinatumomab', SYS_HT_NALM)
sp_48h_blina_hb  = _sp_subset(adata, '48h', 'Blinatumomab', SYS_HT_HB)

all_sp_cols = sp_6h_mock_nm.columns

## CD4 helper synapse — HT/NALM vs HT/HB at 6h Blinatumomab

CD4 helper-T synapse organisation: TCR / CD4 coreceptor plus helper-licensing
costimulators (CD40L=CD154, ICOS=CD278, OX40=CD134, 4-1BB=CD137) in the
**cSMAC**, integrin / SLAM-family adhesion in the **pSMAC**, and bulky
phosphatases (CD45) + mucins (CD43, CD44) in the **exclusion zone**. CD4-
relevant checkpoint receptors (PD-1, TIGIT, CTLA-4, TIM-3) are folded into
the cSMAC since they engage the synapse alongside the TCR.

In [ ]:
# [12 · CD4 helper synapse: heatmaps + networks at 6h Blina−Mock]
# Removed: VISTA, TIGIT, CTLA-4 (CD152), ICOS (CD278), KLRG1
# Added:   CD9, CD81, CD58, CD54  (CD53 already present)
SYNAPSE_MARKERS = [
    'CD3e', 'CD4', 'CD2', 'CD28', 'CD134', 'CD137', 'CD154', 'CD279', 'CD366',
    'CD11a', 'CD50', 'CD48', 'CD352', 'CD53',
    'CD45', 'CD43', 'CD44',
    'CD9', 'CD81', 'CD58', 'CD54',
]
SYNAPSE_CATEGORIES = {'markers': (SYNAPSE_MARKERS, '#888888')}  # single category → no visual divisions

def _delta_sp(sp_num, sp_den):
    """1-row DataFrame whose column-mean equals mean(sp_num) − mean(sp_den) per pair."""
    cols = sp_num.columns.intersection(sp_den.columns)
    delta = sp_num[cols].mean() - sp_den[cols].mean()
    return pd.DataFrame([delta.values], columns=cols)

sp_6h_delta_nm = _delta_sp(sp_6h_blina_nm, sp_6h_mock_nm)
sp_6h_delta_hb = _delta_sp(sp_6h_blina_hb, sp_6h_mock_hb)

mat_syn_nm, mat_syn_hb, mat_syn_diff = plot_synapse_suite(
    sp_a=sp_6h_delta_nm, sp_b=sp_6h_delta_hb,
    categories=SYNAPSE_CATEGORIES,
    label_a='HT/NALM 6h Blina−Mock', label_b='HT/HB 6h Blina−Mock',
    diff_label='ΔΔ (HT/NALM − HT/HB) of (Blina−Mock)',
    suite_name='CD4 helper synapse (Blina−Mock)',
    highlight_node='CD3e',
    var_filter=adata.var_names,
    cluster_k=[3, 5],
)

# CD4 inhibitory signals — HT/NALM vs HT/HB

Focused look at CD4-relevant inhibitory / checkpoint biology:

* **PD-1 (CD279)** — induced on activated CD4, high on Tfh.
* **CTLA-4 (CD152)** — canonical CD4 / Treg checkpoint (constitutive on Tregs, transient on activated CD4).
* **TIGIT** — co-inhibitory, expressed on Treg/Tfh/activated CD4.
* **TIM-3 (CD366)** — Th1 / exhausted-CD4 checkpoint.

VISTA, PSGL-1 (CD162), 2B4 (CD244) and LAIR-1 (CD305) are excluded — they
are more central to myeloid / NK / CD8 biology than to canonical CD4 helper
function.

Sections below:

1. Per-panel violins across time × condition (`[Inh-2]`).
2. Per-system Blina-vs-Mock abundance + spatial top-N (`[Inh-8]`).
3. MA plots (LFC vs mean) per system (`[Inh-8b]`).
4. System-vs-system LFC scatter (`[Inh-8c]`) and spatial coloc diff scatter (`[Inh-8e]`).
5. Heatmap + force-directed network per system + diff (`[Inh-12]`).

In [ ]:
# [Inh-0 · Marker scope — abundance = CD4-relevant inhibitory; spatial pairs require ≥1 inhibitory endpoint]
INHIB_T = ['CD279', 'CD152', 'TIGIT', 'CD366']      # PD-1, CTLA-4, TIGIT, TIM-3

# Pair partners constrained to the inhibitory ∪ CD4 helper-synapse universe.
T_SYNAPSE = ['CD3e', 'CD4', 'CD2', 'CD28', 'CD134', 'CD137', 'CD154', 'CD278',
             'CD279', 'CD152', 'TIGIT', 'CD366',
             'CD11a', 'CD50', 'KLRG1', 'CD48', 'CD352', 'CD53',
             'CD45', 'CD43', 'CD44']

T_PAIR_SCOPE = sorted(set(INHIB_T) | set(T_SYNAPSE))

def _scope_adata(adata_in, abundance_markers, pair_scope, inhib_markers,
                 obsm_key='spatial_asinh5'):
    """Subset var_names to *abundance_markers*; keep obsm pairs whose endpoints are in
    *pair_scope* AND where ≥1 endpoint is in *inhib_markers*."""
    keep = [m for m in abundance_markers if m in adata_in.var_names]
    sub = adata_in[:, keep].copy()
    sp = sub.obsm[obsm_key]
    if not isinstance(sp, pd.DataFrame):
        sp = pd.DataFrame(sp, index=sub.obs_names)
    pair_set = set(pair_scope)
    inhib_set = set(inhib_markers)
    cols = []
    for c in sp.columns:
        endpoints = c.split('/')
        if all(m in pair_set for m in endpoints) and any(m in inhib_set for m in endpoints):
            cols.append(c)
    sub.obsm[obsm_key] = sp[cols].copy()
    return sub

adata_cd4_scope = _scope_adata(adata, INHIB_T, T_PAIR_SCOPE, INHIB_T)

print(f'CD4 scope: abundance {adata_cd4_scope.n_vars} markers '
      f'({list(adata_cd4_scope.var_names)}), '
      f'{adata_cd4_scope.obsm["spatial_asinh5"].shape[1]} spatial pairs (≥1 inhibitory endpoint)')

## Cross-condition comparison — CD4 inhibitory markers across time × condition

In [ ]:
# [Inh-2 · 4-way DA panel: CD4 inhibitory markers — HT/NALM vs HT/HB across time × condition]
mask_nm_cd4 = (
    (adata.obs['cell_system'] == SYS_HT_NALM) &
    (adata.obs['cell_type_annot'] == 'CD4')
)
adata_nm_cd4_inh = adata[mask_nm_cd4].copy()
adata_nm_cd4_inh.obs['time_cond'] = (
    adata_nm_cd4_inh.obs['time'].astype(str) + ' ' + adata_nm_cd4_inh.obs['condition'].astype(str)
)

mask_hb_cd4 = (
    (adata.obs['cell_system'] == SYS_HT_HB) &
    (adata.obs['cell_type_annot'] == 'CD4')
)
adata_hb_cd4_inh = adata[mask_hb_cd4].copy()
adata_hb_cd4_inh.obs['time_cond'] = (
    adata_hb_cd4_inh.obs['time'].astype(str) + ' ' + adata_hb_cd4_inh.obs['condition'].astype(str)
)

print(f'CD4 HT/NALM: {adata_nm_cd4_inh.n_obs}')
print(adata_nm_cd4_inh.obs['time_cond'].value_counts().to_string())
print(f'\nCD4 HT/HB: {adata_hb_cd4_inh.n_obs}')
print(adata_hb_cd4_inh.obs['time_cond'].value_counts().to_string())

plot_marker_panel_violins(
    adata_nm_cd4_inh, 'cd4_inhibitory_markers',
    group_key='time_cond',
    adata_compare=adata_hb_cd4_inh,
    primary_label='HT/NALM',
    compare_label='HT/HB',
)

## Mock vs Blinatumomab — abundance + spatial, inhibitory ∪ synapse, per system

In [ ]:
# [Inh-8 · CD4 Blina vs Mock — abundance + spatial, inhibitory ∪ synapse, per system, 6h]
for sys_label, sys_val in [('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)]:
    plot_condition_comparison(
        adata_cd4_scope,
        time_val='6h',
        cell_system=sys_val,
        cell_type='CD4',
        cond_a='Blinatumomab',
        cond_b='Mock',
        layer='arcsinh',
        obsm_key='spatial_asinh5',
        top_n=15,
        system_label=sys_label,
    )

## MA plots — inhibitory ∪ synapse, CD4, 6h

In [ ]:
# [Inh-8b · MA plots — Blina vs Mock, CD4, inhibitory ∪ synapse, 6h]
ma_results_cd4_inh = plot_ma_blina_vs_mock(
    adata_cd4_scope,
    systems=[('HT/NALM', SYS_HT_NALM), ('HT/HB', SYS_HT_HB)],
    time_val='6h',
    cell_type='CD4',
)

## LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), CD4 inhibitory, 6h

In [ ]:
# [Inh-8c · LFC scatter — Blina vs Mock, HT/HB (x) vs HT/NALM (y), CD4 inhibitory, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_lfc_scatter(
    adata_cd4_scope, time_val='6h',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD4',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd',
    color_below='#2ca02c',
    top_k=4,
    ax=ax,
)
plt.tight_layout()
plt.show()

## Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), CD4 inhibitory pairs, 6h

Mean difference (Blina − Mock) rather than log2 ratio, since coloc values can
be negative (repulsion).

In [ ]:
# [Inh-8e · Spatial coloc diff scatter — Blina − Mock, HT/HB (x) vs HT/NALM (y), CD4 inhibitory pairs, 6h]
fig, ax = plt.subplots(figsize=(8, 8))
plot_spatial_diff_scatter(
    adata_cd4_scope, time_val='6h',
    cond_num='Blinatumomab', cond_den='Mock',
    sys_a=SYS_HT_HB, sys_b=SYS_HT_NALM,
    cell_type='CD4',
    label_a='HT/HB', label_b='HT/NALM',
    color_above='#9467bd', color_below='#2ca02c',
    top_k=10,
    ax=ax,
)
plt.tight_layout()
plt.show()

In [ ]:
# [Export · build full PDF report (markdown + printed outputs + figures)]
import json, re, base64, io, textwrap
from pathlib import Path
from PIL import Image
from matplotlib.backends.backend_pdf import PdfPages

_ANSI = re.compile(r'\x1b\[[0-9;]*m')

def _gather_text(cell):
    parts = []
    for o in cell.get('outputs', []):
        if o.get('output_type') == 'stream':
            parts.append(''.join(o.get('text', [])))
        elif 'text/plain' in o.get('data', {}):
            txt = ''.join(o['data']['text/plain'])
            if txt.lstrip().startswith('<Figure size'):
                continue
            parts.append(txt)
    return _ANSI.sub('', '\n'.join(p.rstrip() for p in parts if p.strip()))

def _gather_images(cell):
    out = []
    for o in cell.get('outputs', []):
        png = o.get('data', {}).get('image/png')
        if png is None:
            continue
        if isinstance(png, list):
            png = ''.join(png)
        out.append(Image.open(io.BytesIO(base64.b64decode(png))).convert('RGB'))
    return out

def _write_text_pages(pdf, text, *, title=None, lines_per_page=62, chars_per_line=100, fontsize=8):
    if not text.strip() and not title:
        return
    raw = []
    for ln in (text.splitlines() or ['']):
        if not ln:
            raw.append('')
        else:
            raw.extend(textwrap.wrap(ln, width=chars_per_line, drop_whitespace=False,
                                     replace_whitespace=False) or [''])
    for start in range(0, max(len(raw), 1), lines_per_page):
        chunk = raw[start:start + lines_per_page]
        fig = plt.figure(figsize=(8.5, 11))
        ax = fig.add_axes([0.05, 0.03, 0.9, 0.94])
        ax.axis('off')
        y = 1.0
        if title and start == 0:
            ax.text(0, y, title, fontsize=11, fontweight='bold', va='top',
                    family='monospace', parse_math=False, transform=ax.transAxes)
            y -= 0.035
        ax.text(0, y, '\n'.join(chunk), fontsize=fontsize, va='top',
                family='monospace', parse_math=False, transform=ax.transAxes)
        pdf.savefig(fig)
        plt.close(fig)

def _write_image_page(pdf, img):
    fig = plt.figure(figsize=(8.5, 11))
    ax = fig.add_subplot(111)
    ax.imshow(img)
    ax.axis('off')
    fig.tight_layout(pad=0.3)
    pdf.savefig(fig)
    plt.close(fig)

def export_notebook_report(notebook_path, out_pdf):
    """Mirror a notebook to PDF: markdown, code-output text (stream/text-plain), and figures."""
    nb_path, out_path = Path(notebook_path), Path(out_pdf)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    nb = json.loads(nb_path.read_text())
    with PdfPages(out_path) as pdf:
        for i, cell in enumerate(nb['cells']):
            if cell['cell_type'] == 'markdown':
                src = ''.join(cell['source']).strip()
                if src:
                    _write_text_pages(pdf, src, title=f'[Cell {i}] markdown')
            else:
                txt = _gather_text(cell)
                if txt:
                    _write_text_pages(pdf, txt, title=f'[Cell {i}] output')
                for img in _gather_images(cell):
                    _write_image_page(pdf, img)
    print(f'wrote {out_path}')
    return out_path


_NB_DIR = Path('/home/projects/nyosef/zvise/PixelGen/PixelGen/New_Data')
export_notebook_report(
    _NB_DIR / 'ht_nalm_vs_ht_hb_cd4_analysis.ipynb',
    _NB_DIR / 'results' / 'ht_nalm_vs_ht_hb_cd4_analysis_report.pdf',
)